# 第16章　货币市场工具与回购市场

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch16_money_market_repo.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch16_money_market_repo.ipynb)

复现例8.1（回购现金流）、例8.2（杠杆套息与负 carry 反噬）与 8.7（杠杆套息策略复盘）。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync` 后再运行本 notebook。')


In [ ]:
import numpy as np
import pandas as pd
from fi import repo, data, plotting
plotting.use_chinese_style()


## 例8.1　7 天质押式回购的现金流


In [ ]:
cf = repo.repo_cashflows(principal=1e8, rate=0.0185, days=7)
print(f"利息       = {cf['interest']:,.2f} 元")
print(f"到期偿还   = {cf['dayN_out']:,.2f} 元")


## 例8.2　杠杆套息的收益与负 carry 反噬


In [ ]:
y, r = 0.0255, 0.0185
for L in (1, 3):
    out = repo.leveraged_carry(y, r, leverage=L)
    print(f'L={L}: carry={out["carry_spread"]*100:.2f}%  ROE={out["roe_annual"]*100:.2f}%')

# 资金面收紧：回购利率升到 2.80%
stress = repo.leveraged_carry(0.0255, 0.0280, leverage=3)
print(f'回购升至2.80%, L=3: carry={stress["carry_spread"]*100:.2f}%  ROE={stress["roe_annual"]*100:.2f}%')
print('haircut 5% -> 理论最大杠杆 =', repo.max_leverage(0.05))


### ROE 关于杠杆的曲线（编程实验 6）

对比两种融资成本，标出负 carry 临界：当 $y<r$ 时，加杠杆反而压低 ROE。


In [ ]:
Ls = np.linspace(1, 5, 81)
roe_easy  = [repo.leveraged_carry(0.0255, 0.0185, L)['roe_annual'] * 100 for L in Ls]
roe_tight = [repo.leveraged_carry(0.0255, 0.0280, L)['roe_annual'] * 100 for L in Ls]

fig, ax = plotting.new_axes()
ax.plot(Ls, roe_easy,  label='融资成本 r=1.85%（正 carry）')
ax.plot(Ls, roe_tight, label='融资成本 r=2.80%（负 carry）')
ax.axhline(2.55, ls=':', color='gray', label='不加杠杆票息 2.55%')
ax.set_xlabel('杠杆 L'); ax.set_ylabel('权益回报 ROE (%)')
ax.set_title('图：ROE 关于杠杆的曲线（正/负 carry）'); ax.legend()
fig.tight_layout()


## 8.7　杠杆套息策略复盘

读取内置货币市场样本：DR007、R007、1Y NCD、10Y 国债收益率（单位 %）。


In [ ]:
mm = data.load_sample('money_market')
mm['date'] = pd.to_datetime(mm['date'])
mm = mm.set_index('date')
mm.head()


### carry 与 R−DR 利差（图16-1、图16-3）


In [ ]:
carry = mm['cgb_10y'] - mm['dr007']        # 持有 10Y 国债、DR007 融资的 carry（%）
r_dr  = mm['r007'] - mm['dr007']           # 非银流动性分层利差（%）

fig, axes = plotting.new_axes(figsize=(9, 6))
fig.clf()
ax1 = fig.add_subplot(2, 1, 1)
ax1.plot(carry.index, carry.values)
ax1.set_ylabel('carry (%)'); ax1.set_title('图16-1　carry = 10Y 国债收益率 − DR007')
ax2 = fig.add_subplot(2, 1, 2)
ax2.plot(r_dr.index, r_dr.values, color='C3')
ax2.set_ylabel('R007 − DR007 (%)'); ax2.set_title('图16-3　非银流动性分层利差')
fig.tight_layout()


### L=1 vs L=3 的累计回报（图16-2）

把每日年化 ROE 折算为日收益累计，比较加杠杆前后的回报与回撤。


In [ ]:
y_daily = mm['cgb_10y'] / 100
r_daily = mm['dr007'] / 100

def daily_roe(L):
    roe_ann = y_daily + (L - 1) * (y_daily - r_daily)   # 年化 ROE 序列
    return roe_ann / 250                                # 折算到每个交易日

cum1 = (1 + daily_roe(1)).cumprod()
cum3 = (1 + daily_roe(3)).cumprod()

def max_drawdown(nav):
    return float(((nav / nav.cummax()) - 1).min())

print(f'L=1: 期末累计 = {cum1.iloc[-1]:.4f}  最大回撤 = {max_drawdown(cum1)*100:.3f}%')
print(f'L=3: 期末累计 = {cum3.iloc[-1]:.4f}  最大回撤 = {max_drawdown(cum3)*100:.3f}%')

fig, ax = plotting.new_axes()
ax.plot(cum1.index, cum1.values, label='L=1（不加杠杆）')
ax.plot(cum3.index, cum3.values, label='L=3')
ax.set_ylabel('累计净值'); ax.set_title('图16-2　杠杆前后累计回报对比'); ax.legend()
fig.tight_layout()


---

> 小结：`fi.repo` 把回购现金流与杠杆套息恒等式 $\text{ROE}=y+(L-1)(y-r)$ 封装为可复用函数；
> 本案例只算 carry，真实损益还需叠加久期资本利得（第6章）与 roll-down（第17章）。
